# Semplice conversione del notebook del punteggio più alto di Michele in una cella combatile con Colab


In [ ]:
# ==============================================================================
# 🏆 AN2DL CHALLENGE 2 - RESNET50 + SVM PIPELINE (v1.2.0 - IMPROVED)
# ==============================================================================
# IMPROVEMENTS:
# - Progressive unfreezing (freeze backbone first, then gradual unfreeze)
# - Discriminative learning rates
# - Stronger regularization
# - SVM hyperparameter tuning with GridSearchCV
# - Cosine annealing LR scheduler
# - Better feature extraction strategy
# ==============================================================================

# --- 0. COLAB-SPECIFIC SETUP ---
!pip install -q gdown lion-pytorch

import os
import cv2
import zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import warnings
import joblib
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import f1_score
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
import gdown
from lion_pytorch import Lion

# --- 1. ENVIRONMENT SETUP ---
warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"⚙️  Hardware: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

set_seed(42)

# --- 2. DATA DOWNLOAD ---
FILES = {
    "crops_data.zip": "1Kit41dsZPNYHNJHml6Wp7JK1yHom1VwK",
    "crops_labels.csv": "17XPlHzQsI4_CUFJletOlVvWRdMuAFkzS",
    "test_data.zip": "1UTRi8b5z_MQSis0IS6L91BqFisTVRgHj"
}

WORK_DIR = "/content"
INPUT_DIR = os.path.join(WORK_DIR, "grumpy-data")

print("\n⬇️  Downloading Dataset Files...")
os.makedirs(INPUT_DIR, exist_ok=True)

for name, fid in FILES.items():
    path = os.path.join(INPUT_DIR, name)
    if not os.path.exists(path):
        gdown.download(f'https://drive.google.com/uc?id={fid}', path, quiet=False)

print("📦  Extracting Data...")
for zfile in ["crops_data.zip", "test_data.zip"]:
    zpath = os.path.join(INPUT_DIR, zfile)
    if os.path.exists(zpath):
        with zipfile.ZipFile(zpath, 'r') as z:
            z.extractall(INPUT_DIR)

# --- 3. PATH CONFIGURATION ---
def find_folder(base, marker_file_ext=".png"):
    for root, _, files in os.walk(base):
        if any(f.endswith(marker_file_ext) for f in files):
            return root
    return base

TRAIN_CROPS_DIR = find_folder(os.path.join(INPUT_DIR, "train_data_crops_crop224_ov56"))
TEST_RAW_DIR = find_folder(os.path.join(INPUT_DIR, "test_data"))
TRAIN_CSV_PATH = os.path.join(INPUT_DIR, "crops_labels.csv")

print(f"\n📂  Training Directory: {TRAIN_CROPS_DIR}")
print(f"📂  Testing Directory:  {TEST_RAW_DIR}")

# --- 4. DATASET ---
class GrumpyDataset(Dataset):
    def __init__(self, mode, df, data_dir, transform=None):
        self.mode = mode
        self.df = df
        self.data_dir = data_dir
        self.transform = transform
        self.map = {'Luminal A': 0, 'Luminal B': 1, 'HER2(+)': 2, 'Triple negative': 3}
        self.norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        fname = self.df.iloc[idx]['sample_index']
        if self.mode == 'train':
            label_str = self.df.iloc[idx]['label']
            target = torch.tensor(self.map[label_str], dtype=torch.long)
        else:
            target = torch.tensor(-1, dtype=torch.long)

        path = os.path.join(self.data_dir, fname)
        img = cv2.imread(path)
        if img is None:
            img = np.zeros((224, 224, 3), dtype=np.uint8)

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = (img / 255.0).astype(np.float32)
        t = torch.from_numpy(img).permute(2, 0, 1)

        if self.transform:
            t = self.transform(t)
        t = self.norm(t)
        return t, target

    def get_raw_test_data(self, idx):
        fname = self.df.iloc[idx]['sample_index']
        img_path = os.path.join(self.data_dir, fname)
        mask_path = img_path.replace("img_", "mask_")

        img = cv2.imread(img_path)
        if img is None:
            return None, None
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = (img / 255.0).astype(np.float32)

        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if mask is None:
            mask = np.zeros(img.shape[:2], dtype=np.uint8)
        _, mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
        return img, mask

# --- 5. DATA PREPARATION ---
print("📊  Configuring Patient-Aware Stratified Split...")
df_full = pd.read_csv(TRAIN_CSV_PATH)
df_full['patient_id'] = df_full['sample_index'].apply(lambda x: '_'.join(x.split('_')[:2]))

unique_patients = df_full.drop_duplicates(subset='patient_id')[['patient_id', 'label']]
train_patients, val_patients = train_test_split(
    unique_patients['patient_id'],
    test_size=0.2,
    stratify=unique_patients['label'],
    random_state=42
)

train_df = df_full[df_full['patient_id'].isin(train_patients)].reset_index(drop=True)
val_df = df_full[df_full['patient_id'].isin(val_patients)].reset_index(drop=True)

# Stronger augmentation
train_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=90),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.15),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.1)),  # Cutout-like augmentation
])

train_loader = DataLoader(
    GrumpyDataset('train', train_df, TRAIN_CROPS_DIR, train_tf),
    batch_size=32, shuffle=True, num_workers=0, drop_last=True
)
val_loader = DataLoader(
    GrumpyDataset('train', val_df, TRAIN_CROPS_DIR, None),
    batch_size=32, shuffle=False, num_workers=0
)
train_feat_loader = DataLoader(
    GrumpyDataset('train', train_df, TRAIN_CROPS_DIR, None),
    batch_size=32, shuffle=False, num_workers=0
)
val_feat_loader = DataLoader(
    GrumpyDataset('train', val_df, TRAIN_CROPS_DIR, None),
    batch_size=32, shuffle=False, num_workers=0
)

print(f"   Train Set: {len(train_df)} patches | Val Set: {len(val_df)} patches")

# --- 6. MODEL ARCHITECTURE ---
class ResNet50FeatureExtractor(nn.Module):
    def __init__(self, num_classes=4, feature_dim=512, dropout=0.4):
        super().__init__()
        base = resnet50(weights=ResNet50_Weights.DEFAULT)

        # Split backbone into stages for progressive unfreezing
        self.layer0 = nn.Sequential(base.conv1, base.bn1, base.relu, base.maxpool)
        self.layer1 = base.layer1
        self.layer2 = base.layer2
        self.layer3 = base.layer3
        self.layer4 = base.layer4
        self.avgpool = base.avgpool

        self.feature_dim = base.fc.in_features  # 2048

        # Projection head with stronger regularization
        self.projection = nn.Sequential(
            nn.Linear(self.feature_dim, feature_dim),
            nn.BatchNorm1d(feature_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        # Classification head
        self.classifier = nn.Linear(feature_dim, num_classes)

    def get_backbone_params(self):
        """Return backbone parameters grouped by stage."""
        return {
            'early': list(self.layer0.parameters()) + list(self.layer1.parameters()) + list(self.layer2.parameters()),
            'late': list(self.layer3.parameters()) + list(self.layer4.parameters()),
        }

    def freeze_backbone(self):
        """Freeze all backbone layers."""
        for param in self.layer0.parameters():
            param.requires_grad = False
        for param in self.layer1.parameters():
            param.requires_grad = False
        for param in self.layer2.parameters():
            param.requires_grad = False
        for param in self.layer3.parameters():
            param.requires_grad = False
        for param in self.layer4.parameters():
            param.requires_grad = False

    def unfreeze_late_layers(self):
        """Unfreeze layer3 and layer4."""
        for param in self.layer3.parameters():
            param.requires_grad = True
        for param in self.layer4.parameters():
            param.requires_grad = True

    def unfreeze_all(self):
        """Unfreeze all layers."""
        for param in self.parameters():
            param.requires_grad = True

    def forward(self, x, return_features=False):
        x = self.layer0(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)

        proj_feat = self.projection(x)

        if return_features:
            return proj_feat

        return self.classifier(proj_feat)

    def extract_features(self, x):
        return self.forward(x, return_features=True)


model = ResNet50FeatureExtractor(num_classes=4, feature_dim=512, dropout=0.4).to(device)

# --- 7. PROGRESSIVE TRAINING ---
MODEL_SAVE_PATH = "/content/best_backbone_v120.pth"
SVM_SAVE_PATH = "/content/svm_classifier.joblib"
SCALER_SAVE_PATH = "/content/feature_scaler.joblib"

weights = torch.tensor([1.65, 1.2, 1.7, 4.0]).to(device)
criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.1)

best_val_f1 = -1.0
scaler = torch.cuda.amp.GradScaler()

def train_epoch(model, loader, optimizer, criterion, scaler, use_mixup=True):
    model.train()
    running_loss = 0.0
    pbar = tqdm(loader, desc="Training", leave=False)

    for inputs, targets in pbar:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()

        do_mixup = use_mixup and np.random.rand() < 0.5
        if do_mixup:
            lam = np.random.beta(0.4, 0.4)  # Stronger mixup
            idx = torch.randperm(inputs.size(0)).to(device)
            mixed_input = lam * inputs + (1 - lam) * inputs[idx]
            t_a, t_b = targets, targets[idx]

        with torch.cuda.amp.autocast():
            if do_mixup:
                out = model(mixed_input)
                loss = lam * criterion(out, t_a) + (1 - lam) * criterion(out, t_b)
            else:
                out = model(inputs)
                loss = criterion(out, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item()

    return running_loss / len(loader)

def validate(model, loader):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            with torch.cuda.amp.autocast():
                out = model(inputs)
            preds.extend(out.argmax(1).cpu().numpy())
            trues.extend(targets.cpu().numpy())
    return f1_score(trues, preds, average='micro')

# ==================== PHASE 1: Train only new layers ====================
print("\n" + "="*60)
print("🔒 PHASE 1: Training projection + classifier (backbone frozen)")
print("="*60)

model.freeze_backbone()

optimizer = Lion(
    list(model.projection.parameters()) + list(model.classifier.parameters()),
    lr=3e-4, weight_decay=1e-3
)
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2)

PHASE1_EPOCHS = 15
patience_counter = 0
PATIENCE = 8

for epoch in range(1, PHASE1_EPOCHS + 1):
    loss = train_epoch(model, train_loader, optimizer, criterion, scaler, use_mixup=True)
    val_f1 = validate(model, val_loader)
    scheduler.step()

    print(f"   Ep {epoch} | Loss: {loss:.4f} | Val F1: {val_f1:.4f} | LR: {scheduler.get_last_lr()[0]:.2e}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        patience_counter = 0
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print("   💾 Best Model Saved")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print("   ⏹️  Early stopping Phase 1")
            break

# ==================== PHASE 2: Unfreeze late layers ====================
print("\n" + "="*60)
print("🔓 PHASE 2: Fine-tuning late backbone layers (layer3, layer4)")
print("="*60)

model.load_state_dict(torch.load(MODEL_SAVE_PATH))
model.unfreeze_late_layers()

optimizer = Lion([
    {'params': model.layer3.parameters(), 'lr': 1e-5},
    {'params': model.layer4.parameters(), 'lr': 1e-5},
    {'params': model.projection.parameters(), 'lr': 1e-4},
    {'params': model.classifier.parameters(), 'lr': 1e-4},
], weight_decay=1e-3)
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2)

PHASE2_EPOCHS = 20
patience_counter = 0
PATIENCE = 10

for epoch in range(1, PHASE2_EPOCHS + 1):
    loss = train_epoch(model, train_loader, optimizer, criterion, scaler, use_mixup=True)
    val_f1 = validate(model, val_loader)
    scheduler.step()

    print(f"   Ep {epoch} | Loss: {loss:.4f} | Val F1: {val_f1:.4f} | LR: {scheduler.get_last_lr()[0]:.2e}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        patience_counter = 0
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print("   💾 Best Model Saved")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print("   ⏹️  Early stopping Phase 2")
            break

# ==================== PHASE 3: Full fine-tuning (optional) ====================
print("\n" + "="*60)
print("🔓 PHASE 3: Full fine-tuning (all layers)")
print("="*60)

model.load_state_dict(torch.load(MODEL_SAVE_PATH))
model.unfreeze_all()

optimizer = Lion([
    {'params': model.layer0.parameters(), 'lr': 1e-6},
    {'params': model.layer1.parameters(), 'lr': 1e-6},
    {'params': model.layer2.parameters(), 'lr': 5e-6},
    {'params': model.layer3.parameters(), 'lr': 1e-5},
    {'params': model.layer4.parameters(), 'lr': 1e-5},
    {'params': model.projection.parameters(), 'lr': 5e-5},
    {'params': model.classifier.parameters(), 'lr': 5e-5},
], weight_decay=1e-3)
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=3, T_mult=2)

PHASE3_EPOCHS = 15
patience_counter = 0
PATIENCE = 8

for epoch in range(1, PHASE3_EPOCHS + 1):
    loss = train_epoch(model, train_loader, optimizer, criterion, scaler, use_mixup=True)
    val_f1 = validate(model, val_loader)
    scheduler.step()

    print(f"   Ep {epoch} | Loss: {loss:.4f} | Val F1: {val_f1:.4f} | LR: {scheduler.get_last_lr()[0]:.2e}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        patience_counter = 0
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print("   💾 Best Model Saved")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print("   ⏹️  Early stopping Phase 3")
            break

print(f"\n🏆 Best Validation F1: {best_val_f1:.4f}")

# --- 8. FEATURE EXTRACTION ---
print("\n🔬  Extracting Features for SVM...")

model.load_state_dict(torch.load(MODEL_SAVE_PATH))
model.eval()

def extract_all_features(loader, model, device):
    all_features, all_labels = [], []
    with torch.no_grad():
        for inputs, targets in tqdm(loader, desc="Extracting", leave=False):
            inputs = inputs.to(device)
            with torch.cuda.amp.autocast():
                features = model.extract_features(inputs)
            all_features.append(features.cpu().numpy())
            all_labels.append(targets.numpy())
    return np.vstack(all_features), np.concatenate(all_labels)

X_train, y_train = extract_all_features(train_feat_loader, model, device)
X_val, y_val = extract_all_features(val_feat_loader, model, device)

print(f"   Train features: {X_train.shape} | Val features: {X_val.shape}")

# --- 9. SVM WITH GRID SEARCH ---
print("\n🎯  Training SVM with GridSearchCV...")

scaler_feat = StandardScaler()
X_train_scaled = scaler_feat.fit_transform(X_train)
X_val_scaled = scaler_feat.transform(X_val)

# Combine train and val for cross-validation
X_all = np.vstack([X_train_scaled, X_val_scaled])
y_all = np.concatenate([y_train, y_val])

param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.01, 0.001],
    'kernel': ['rbf', 'poly'],
}

svm = SVC(class_weight='balanced', probability=True, random_state=42)
grid_search = GridSearchCV(
    svm, param_grid, cv=5, scoring='f1_micro', n_jobs=-1, verbose=1
)
grid_search.fit(X_all, y_all)

print(f"\n✅ Best SVM Parameters: {grid_search.best_params_}")
print(f"✅ Best CV F1 Score: {grid_search.best_score_:.4f}")

svm_classifier = grid_search.best_estimator_

# Final evaluation on validation set
val_preds_svm = svm_classifier.predict(X_val_scaled)
val_f1_svm = f1_score(y_val, val_preds_svm, average='micro')
print(f"✅ SVM Validation F1: {val_f1_svm:.4f}")

joblib.dump(svm_classifier, SVM_SAVE_PATH)
joblib.dump(scaler_feat, SCALER_SAVE_PATH)

# --- 10. INFERENCE ---
print("\n🔮  Running Inference...")

model.load_state_dict(torch.load(MODEL_SAVE_PATH))
model.eval()
svm_classifier = joblib.load(SVM_SAVE_PATH)
scaler_feat = joblib.load(SCALER_SAVE_PATH)

norm_mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
norm_std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
inv_map = {0: 'Luminal A', 1: 'Luminal B', 2: 'HER2(+)', 3: 'Triple negative'}

test_files = sorted([f for f in os.listdir(TEST_RAW_DIR) if f.startswith('img_') and f.endswith('.png')])
df_test = pd.DataFrame({'sample_index': test_files})
ds_test = GrumpyDataset('test', df_test, TEST_RAW_DIR)

results = []

for idx in tqdm(range(len(ds_test))):
    fname = ds_test.df.iloc[idx]['sample_index']
    full_img, full_mask = ds_test.get_raw_test_data(idx)

    if full_img is None:
        results.append({'sample_index': fname, 'label': 'Luminal B'})
        continue

    H, W, _ = full_img.shape
    patches = []

    for y in range(0, H - 224 + 1, 112):
        for x in range(0, W - 224 + 1, 112):
            mask_patch = full_mask[y:y+224, x:x+224]
            if np.sum(mask_patch > 0) > (224 * 224 * 0.1):
                p = full_img[y:y+224, x:x+224]
                p_t = torch.from_numpy(p).permute(2, 0, 1)
                p_t = (p_t - norm_mean) / norm_std
                patches.append(p_t)

    pred_label = "Luminal B"

    if len(patches) > 0:
        batch = torch.stack(patches).to(device)

        features_list = []
        with torch.no_grad():
            for i in range(0, len(batch), 32):
                with torch.cuda.amp.autocast():
                    feats = model.extract_features(batch[i:i+32])
                features_list.append(feats.cpu().numpy())

        all_features = np.vstack(features_list)
        all_features_scaled = scaler_feat.transform(all_features)

        probs = svm_classifier.predict_proba(all_features_scaled)

        # Top-K voting (top 50%)
        k = max(1, int(len(probs) * 0.3))
        max_probs = np.max(probs, axis=1)
        topk_indices = np.argsort(max_probs)[-k:]
        avg_probs = probs[topk_indices].mean(axis=0)

        pred_label = inv_map[np.argmax(avg_probs)]

    results.append({'sample_index': fname, 'label': pred_label})

SUBMISSION_PATH = "/content/submission.csv"
pd.DataFrame(results).to_csv(SUBMISSION_PATH, index=False)
print(f"✅ Done! {SUBMISSION_PATH} generated.")

from google.colab import files
files.download(SUBMISSION_PATH)

⚙️  Hardware: Tesla T4

⬇️  Downloading Dataset Files...
📦  Extracting Data...

📂  Training Directory: /content/grumpy-data/train_data_crops_crop224_ov56
📂  Testing Directory:  /content/grumpy-data/test_data
📊  Configuring Patient-Aware Stratified Split...
   Train Set: 2872 patches | Val Set: 690 patches

🔒 PHASE 1: Training projection + classifier (backbone frozen)


Training:   0%|          | 0/89 [00:00<?, ?it/s]

   Ep 1 | Loss: 1.5017 | Val F1: 0.2797 | LR: 2.71e-04
   💾 Best Model Saved


Training:   0%|          | 0/89 [00:00<?, ?it/s]

   Ep 2 | Loss: 1.3823 | Val F1: 0.3304 | LR: 1.96e-04
   💾 Best Model Saved


...

   Train features: (2872, 512) | Val features: (690, 512)

🎯  Training SVM with GridSearchCV...
Fitting 5 folds for each of 32 candidates, totalling 160 fits

✅ Best SVM Parameters: {'C': 0.1, 'gamma': 0.001, 'kernel': 'rbf'}
✅ Best CV F1 Score: 0.5435
✅ SVM Validation F1: 0.4522

🔮  Running Inference...


  0%|          | 0/477 [00:00<?, ?it/s]

✅ Done! /content/submission.csv generated.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# EfficientNet-B3 + SVM, patient-split, MixUp, Lion, weighted CE (LS=0.15), 2-phase freeze/finetune, mask-tiling (10%), top-50% voting


In [ ]:
# ==============================================================================
# 🏆 AN2DL CHALLENGE 2 - MULTI-ARCHITECTURE + SVM PIPELINE (v1.3.0)
# ==============================================================================
# NEW FEATURES:
# - Multiple backbone options: EfficientNet-B3, ConvNeXt-Tiny, ResNet50
# - Ensemble prediction option
# - Optimized for histopathology
# ==============================================================================

# --- 0. SETUP ---
!pip install -q gdown lion-pytorch timm

import os
import cv2
import zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import warnings
import joblib
import timm
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import f1_score
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
import gdown
from lion_pytorch import Lion

# ==============================================================================
# 🔧 CONFIGURATION - CHANGE THESE TO EXPERIMENT
# ==============================================================================
CONFIG = {
    # Architecture: 'efficientnet_b3', 'convnext_tiny', 'resnet50', 'swin_tiny', 'vit_small'
    'backbone': 'efficientnet_b3',

    # Feature dimension for projection head
    'feature_dim': 256,

    # Regularization
    'dropout': 0.5,
    'weight_decay': 1e-2,
    'label_smoothing': 0.15,

    # Training
    'batch_size': 32,
    'phase1_epochs': 20,
    'phase2_epochs': 25,
    'phase1_lr': 3e-4,
    'phase2_lr': 1e-5,

    # SVM
    'use_svm': True,  # If False, use neural network classifier only

    # Seed
    'seed': 42
}

print(f"🔧 Configuration: {CONFIG['backbone']} backbone")

# --- 1. ENVIRONMENT ---
warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"⚙️  Hardware: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

def set_seed(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

set_seed(CONFIG['seed'])

# --- 2. DATA DOWNLOAD ---
FILES = {
    "crops_data.zip": "1Kit41dsZPNYHNJHml6Wp7JK1yHom1VwK",
    "crops_labels.csv": "17XPlHzQsI4_CUFJletOlVvWRdMuAFkzS",
    "test_data.zip": "1UTRi8b5z_MQSis0IS6L91BqFisTVRgHj"
}

WORK_DIR = "/content"
INPUT_DIR = os.path.join(WORK_DIR, "grumpy-data")

print("\n⬇️  Downloading Dataset Files...")
os.makedirs(INPUT_DIR, exist_ok=True)

for name, fid in FILES.items():
    path = os.path.join(INPUT_DIR, name)
    if not os.path.exists(path):
        gdown.download(f'https://drive.google.com/uc?id={fid}', path, quiet=False)

print("📦  Extracting Data...")
for zfile in ["crops_data.zip", "test_data.zip"]:
    zpath = os.path.join(INPUT_DIR, zfile)
    if os.path.exists(zpath):
        with zipfile.ZipFile(zpath, 'r') as z:
            z.extractall(INPUT_DIR)

def find_folder(base, marker_file_ext=".png"):
    for root, _, files in os.walk(base):
        if any(f.endswith(marker_file_ext) for f in files):
            return root
    return base

TRAIN_CROPS_DIR = find_folder(os.path.join(INPUT_DIR, "train_data_crops_crop224_ov56"))
TEST_RAW_DIR = find_folder(os.path.join(INPUT_DIR, "test_data"))
TRAIN_CSV_PATH = os.path.join(INPUT_DIR, "crops_labels.csv")

print(f"\n📂  Training: {TRAIN_CROPS_DIR}")
print(f"📂  Testing:  {TEST_RAW_DIR}")

# --- 3. DATASET ---
class GrumpyDataset(Dataset):
    def __init__(self, mode, df, data_dir, transform=None):
        self.mode = mode
        self.df = df
        self.data_dir = data_dir
        self.transform = transform
        self.map = {'Luminal A': 0, 'Luminal B': 1, 'HER2(+)': 2, 'Triple negative': 3}
        self.norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        fname = self.df.iloc[idx]['sample_index']
        target = torch.tensor(
            self.map[self.df.iloc[idx]['label']] if self.mode == 'train' else -1,
            dtype=torch.long
        )

        img = cv2.imread(os.path.join(self.data_dir, fname))
        if img is None:
            img = np.zeros((224, 224, 3), dtype=np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = (img / 255.0).astype(np.float32)

        t = torch.from_numpy(img).permute(2, 0, 1)
        if self.transform:
            t = self.transform(t)
        t = self.norm(t)
        return t, target

    def get_raw_test_data(self, idx):
        fname = self.df.iloc[idx]['sample_index']
        img_path = os.path.join(self.data_dir, fname)
        mask_path = img_path.replace("img_", "mask_")

        img = cv2.imread(img_path)
        if img is None:
            return None, None
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = (img / 255.0).astype(np.float32)

        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if mask is None:
            mask = np.zeros(img.shape[:2], dtype=np.uint8)
        _, mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
        return img, mask

# --- 4. DATA PREPARATION ---
print("📊  Preparing Data...")
df_full = pd.read_csv(TRAIN_CSV_PATH)
df_full['patient_id'] = df_full['sample_index'].apply(lambda x: '_'.join(x.split('_')[:2]))

unique_patients = df_full.drop_duplicates(subset='patient_id')[['patient_id', 'label']]
train_patients, val_patients = train_test_split(
    unique_patients['patient_id'],
    test_size=0.2,
    stratify=unique_patients['label'],
    random_state=CONFIG['seed']
)

train_df = df_full[df_full['patient_id'].isin(train_patients)].reset_index(drop=True)
val_df = df_full[df_full['patient_id'].isin(val_patients)].reset_index(drop=True)

# Strong augmentation for histopathology
train_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomApply([transforms.RandomRotation(degrees=(90, 90))], p=0.5),
    transforms.RandomApply([transforms.RandomRotation(degrees=(180, 180))], p=0.5),
    transforms.RandomApply([transforms.RandomRotation(degrees=(270, 270))], p=0.5),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3)], p=0.2),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.15)),
])

train_loader = DataLoader(
    GrumpyDataset('train', train_df, TRAIN_CROPS_DIR, train_tf),
    batch_size=CONFIG['batch_size'], shuffle=True, num_workers=0, drop_last=True
)
val_loader = DataLoader(
    GrumpyDataset('train', val_df, TRAIN_CROPS_DIR, None),
    batch_size=CONFIG['batch_size'], shuffle=False, num_workers=0
)
train_feat_loader = DataLoader(
    GrumpyDataset('train', train_df, TRAIN_CROPS_DIR, None),
    batch_size=CONFIG['batch_size'], shuffle=False, num_workers=0
)

print(f"   Train: {len(train_df)} | Val: {len(val_df)} patches")

# --- 5. MODEL ARCHITECTURE ---
class FlexibleFeatureExtractor(nn.Module):
    """
    Flexible feature extractor supporting multiple backbones.
    """
    def __init__(self, backbone_name, num_classes=4, feature_dim=256, dropout=0.5):
        super().__init__()
        self.backbone_name = backbone_name

        # Create backbone using timm
        self.backbone = timm.create_model(
            backbone_name,
            pretrained=True,
            num_classes=0,  # Remove classifier
            global_pool='avg'
        )

        # Get feature dimension
        with torch.no_grad():
            dummy = torch.randn(1, 3, 224, 224)
            backbone_dim = self.backbone(dummy).shape[1]

        print(f"   Backbone: {backbone_name} | Features: {backbone_dim} → {feature_dim}")

        # Projection head
        self.projection = nn.Sequential(
            nn.Linear(backbone_dim, feature_dim),
            nn.BatchNorm1d(feature_dim),
            nn.GELU(),  # GELU often better than ReLU for transformers/modern CNNs
            nn.Dropout(dropout),
        )

        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(feature_dim, feature_dim // 2),
            nn.BatchNorm1d(feature_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(feature_dim // 2, num_classes)
        )

        self.backbone_dim = backbone_dim
        self.feature_dim = feature_dim

    def freeze_backbone(self):
        for param in self.backbone.parameters():
            param.requires_grad = False

    def unfreeze_backbone(self):
        for param in self.backbone.parameters():
            param.requires_grad = True

    def forward(self, x, return_features=False):
        feat = self.backbone(x)
        proj = self.projection(feat)

        if return_features:
            return proj
        return self.classifier(proj)

    def extract_features(self, x):
        return self.forward(x, return_features=True)

    def get_trainable_params(self, phase='head'):
        """Get parameters for different training phases."""
        if phase == 'head':
            return list(self.projection.parameters()) + list(self.classifier.parameters())
        elif phase == 'full':
            return self.parameters()
        elif phase == 'backbone_slow':
            # Backbone with lower LR, head with higher LR
            return [
                {'params': self.backbone.parameters(), 'lr': CONFIG['phase2_lr']},
                {'params': self.projection.parameters(), 'lr': CONFIG['phase2_lr'] * 10},
                {'params': self.classifier.parameters(), 'lr': CONFIG['phase2_lr'] * 10},
            ]

# Backbone mapping
BACKBONE_MAP = {
    'efficientnet_b3': 'efficientnet_b3',
    'efficientnet_b2': 'efficientnet_b2',
    'convnext_tiny': 'convnext_tiny',
    'convnext_small': 'convnext_small',
    'resnet50': 'resnet50',
    'swin_tiny': 'swin_tiny_patch4_window7_224',
    'swin_small': 'swin_small_patch4_window7_224',
    'vit_small': 'vit_small_patch16_224',
    'vit_base': 'vit_base_patch16_224',
}

backbone_name = BACKBONE_MAP.get(CONFIG['backbone'], CONFIG['backbone'])
model = FlexibleFeatureExtractor(
    backbone_name=backbone_name,
    num_classes=4,
    feature_dim=CONFIG['feature_dim'],
    dropout=CONFIG['dropout']
).to(device)

# --- 6. TRAINING UTILITIES ---
MODEL_SAVE_PATH = f"/content/best_{CONFIG['backbone']}.pth"
SVM_SAVE_PATH = "/content/svm_classifier.joblib"
SCALER_SAVE_PATH = "/content/feature_scaler.joblib"

weights = torch.tensor([1.65, 1.2, 1.7, 4.0]).to(device)
criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=CONFIG['label_smoothing'])
scaler = torch.cuda.amp.GradScaler()

best_val_f1 = -1.0

def train_epoch(model, loader, optimizer, criterion, use_mixup=True, mixup_alpha=0.4):
    model.train()
    running_loss = 0.0

    for inputs, targets in tqdm(loader, desc="Training", leave=False):
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()

        # MixUp augmentation
        if use_mixup and np.random.rand() < 0.5:
            lam = np.random.beta(mixup_alpha, mixup_alpha)
            idx = torch.randperm(inputs.size(0)).to(device)
            inputs = lam * inputs + (1 - lam) * inputs[idx]
            targets_a, targets_b = targets, targets[idx]

            with torch.cuda.amp.autocast():
                out = model(inputs)
                loss = lam * criterion(out, targets_a) + (1 - lam) * criterion(out, targets_b)
        else:
            with torch.cuda.amp.autocast():
                out = model(inputs)
                loss = criterion(out, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item()

    return running_loss / len(loader)

def validate(model, loader):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for inputs, targets in loader:
            inputs = inputs.to(device)
            with torch.cuda.amp.autocast():
                out = model(inputs)
            preds.extend(out.argmax(1).cpu().numpy())
            trues.extend(targets.numpy())
    return f1_score(trues, preds, average='micro')

def train_phase(model, train_loader, val_loader, optimizer, scheduler, epochs, patience, phase_name):
    global best_val_f1
    patience_counter = 0

    print(f"\n{'='*60}")
    print(f"📍 {phase_name}")
    print(f"{'='*60}")

    for epoch in range(1, epochs + 1):
        loss = train_epoch(model, train_loader, optimizer, criterion)
        val_f1 = validate(model, val_loader)
        if scheduler:
            scheduler.step()

        lr_str = f"{optimizer.param_groups[0]['lr']:.2e}"
        print(f"   Ep {epoch:2d} | Loss: {loss:.4f} | Val F1: {val_f1:.4f} | LR: {lr_str}")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            patience_counter = 0
            torch.save(model.state_dict(), MODEL_SAVE_PATH)
            print("   💾 Best Model Saved")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"   ⏹️  Early stopping")
                break

    return best_val_f1

# --- 7. PHASE 1: Train head only (backbone frozen) ---
model.freeze_backbone()

optimizer = Lion(
    model.get_trainable_params('head'),
    lr=CONFIG['phase1_lr'],
    weight_decay=CONFIG['weight_decay']
)
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2)

train_phase(
    model, train_loader, val_loader, optimizer, scheduler,
    epochs=CONFIG['phase1_epochs'], patience=10,
    phase_name="PHASE 1: Training Head (Backbone Frozen)"
)

# --- 8. PHASE 2: Fine-tune full model ---
model.load_state_dict(torch.load(MODEL_SAVE_PATH))
model.unfreeze_backbone()

optimizer = Lion(
    model.get_trainable_params('backbone_slow'),
    weight_decay=CONFIG['weight_decay']
)
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2)

train_phase(
    model, train_loader, val_loader, optimizer, scheduler,
    epochs=CONFIG['phase2_epochs'], patience=12,
    phase_name="PHASE 2: Fine-tuning Full Model"
)

print(f"\n🏆 Best Neural Network F1: {best_val_f1:.4f}")

# --- 9. SVM CLASSIFIER (Optional) ---
if CONFIG['use_svm']:
    print("\n🔬  Extracting Features for SVM...")

    model.load_state_dict(torch.load(MODEL_SAVE_PATH))
    model.eval()

    def extract_features(loader):
        features, labels = [], []
        with torch.no_grad():
            for inputs, targets in tqdm(loader, leave=False):
                inputs = inputs.to(device)
                with torch.cuda.amp.autocast():
                    feat = model.extract_features(inputs)
                features.append(feat.cpu().numpy())
                labels.append(targets.numpy())
        return np.vstack(features), np.concatenate(labels)

    X_train, y_train = extract_features(train_feat_loader)
    X_val, y_val = extract_features(val_loader)

    print(f"   Features: {X_train.shape}")

    # Scale features
    scaler_feat = StandardScaler()
    X_train_scaled = scaler_feat.fit_transform(X_train)
    X_val_scaled = scaler_feat.transform(X_val)

    # Grid search for SVM
    print("\n🎯  SVM GridSearchCV...")

    X_all = np.vstack([X_train_scaled, X_val_scaled])
    y_all = np.concatenate([y_train, y_val])

    param_grid = {
        'C': [0.1, 1, 10, 50],
        'gamma': ['scale', 0.01, 0.001],
        'kernel': ['rbf'],
    }

    grid = GridSearchCV(
        SVC(class_weight='balanced', probability=True, random_state=42),
        param_grid, cv=5, scoring='f1_micro', n_jobs=-1, verbose=1
    )
    grid.fit(X_all, y_all)

    svm_classifier = grid.best_estimator_
    print(f"\n✅ Best SVM: {grid.best_params_}")
    print(f"✅ Best CV F1: {grid.best_score_:.4f}")

    # Evaluate
    val_preds = svm_classifier.predict(X_val_scaled)
    svm_f1 = f1_score(y_val, val_preds, average='micro')
    print(f"✅ SVM Val F1: {svm_f1:.4f}")

    joblib.dump(svm_classifier, SVM_SAVE_PATH)
    joblib.dump(scaler_feat, SCALER_SAVE_PATH)

# --- 10. INFERENCE ---
print("\n🔮  Running Inference...")

model.load_state_dict(torch.load(MODEL_SAVE_PATH))
model.eval()

if CONFIG['use_svm']:
    svm_classifier = joblib.load(SVM_SAVE_PATH)
    scaler_feat = joblib.load(SCALER_SAVE_PATH)

norm_mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
norm_std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
inv_map = {0: 'Luminal A', 1: 'Luminal B', 2: 'HER2(+)', 3: 'Triple negative'}

test_files = sorted([f for f in os.listdir(TEST_RAW_DIR) if f.startswith('img_') and f.endswith('.png')])
df_test = pd.DataFrame({'sample_index': test_files})
ds_test = GrumpyDataset('test', df_test, TEST_RAW_DIR)

results = []

for idx in tqdm(range(len(ds_test))):
    fname = ds_test.df.iloc[idx]['sample_index']
    full_img, full_mask = ds_test.get_raw_test_data(idx)

    if full_img is None:
        results.append({'sample_index': fname, 'label': 'Luminal B'})
        continue

    H, W, _ = full_img.shape
    patches = []

    # Extract patches with tissue
    for y in range(0, H - 224 + 1, 112):
        for x in range(0, W - 224 + 1, 112):
            mask_patch = full_mask[y:y+224, x:x+224]
            if np.sum(mask_patch > 0) > (224 * 224 * 0.1):
                p = full_img[y:y+224, x:x+224]
                p_t = torch.from_numpy(p).permute(2, 0, 1)
                p_t = (p_t - norm_mean) / norm_std
                patches.append(p_t)

    pred_label = "Luminal B"

    if len(patches) > 0:
        batch = torch.stack(patches).to(device)

        if CONFIG['use_svm']:
            # SVM prediction
            features_list = []
            with torch.no_grad():
                for i in range(0, len(batch), 32):
                    with torch.cuda.amp.autocast():
                        feats = model.extract_features(batch[i:i+32])
                    features_list.append(feats.cpu().numpy())

            all_features = scaler_feat.transform(np.vstack(features_list))
            probs = svm_classifier.predict_proba(all_features)
        else:
            # Neural network prediction
            probs_list = []
            with torch.no_grad():
                for i in range(0, len(batch), 32):
                    with torch.cuda.amp.autocast():
                        logits = model(batch[i:i+32])
                    probs_list.append(F.softmax(logits, dim=1).cpu().numpy())
            probs = np.vstack(probs_list)

        # Top-K voting
        k = max(1, int(len(probs) * 0.5))
        max_probs = np.max(probs, axis=1)
        topk_idx = np.argsort(max_probs)[-k:]
        avg_probs = probs[topk_idx].mean(axis=0)

        pred_label = inv_map[np.argmax(avg_probs)]

    results.append({'sample_index': fname, 'label': pred_label})

# Save results
SUBMISSION_PATH = "/content/submission.csv"
pd.DataFrame(results).to_csv(SUBMISSION_PATH, index=False)
print(f"\n✅ Done! Saved to {SUBMISSION_PATH}")

# Summary
print("\n" + "="*60)
print("📊 SUMMARY")
print("="*60)
print(f"   Backbone:     {CONFIG['backbone']}")
print(f"   Best NN F1:   {best_val_f1:.4f}")
if CONFIG['use_svm']:
    print(f"   SVM CV F1:    {grid.best_score_:.4f}")
print("="*60)

from google.colab import files
files.download(SUBMISSION_PATH)

🔧 Configuration: efficientnet_b3 backbone
⚙️  Hardware: Tesla T4

⬇️  Downloading Dataset Files...
📦  Extracting Data...

📂  Training: /content/grumpy-data/train_data_crops_crop224_ov56
📂  Testing:  /content/grumpy-data/test_data
📊  Preparing Data...
   Train: 2872 | Val: 690 patches


model.safetensors:   0%|          | 0.00/49.3M [00:00<?, ?B/s]

   Backbone: efficientnet_b3 | Features: 1536 → 256

📍 PHASE 1: Training Head (Backbone Frozen)


Training:   0%|          | 0/89 [00:00<?, ?it/s]

   Ep  1 | Loss: 1.4756 | Val F1: 0.2362 | LR: 2.71e-04
   💾 Best Model Saved


...

   Features: (2872, 256)

🎯  SVM GridSearchCV...
Fitting 5 folds for each of 12 candidates, totalling 60 fits

✅ Best SVM: {'C': 0.1, 'gamma': 'scale', 'kernel': 'rbf'}
✅ Best CV F1: 0.5645
✅ SVM Val F1: 0.4029

🔮  Running Inference...


  0%|          | 0/477 [00:00<?, ?it/s]


✅ Done! Saved to /content/submission.csv

📊 SUMMARY
   Backbone:     efficientnet_b3
   Best NN F1:   0.3333
   SVM CV F1:    0.5645


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>